# CUDA GPU Test

Verifies TensorFlow detects the GPU and can run a simple computation.

In [1]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('Built with CUDA:', tf.test.is_built_with_cuda())

I0000 00:00:1780355164.115278   29450 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.21.0
Built with CUDA: True


In [2]:
gpus = tf.config.list_physical_devices('GPU')
print('GPUs disponibles:', gpus)
if gpus:
    print('GPU detectada:', gpus[0].name)
    print('Tipo de device:', gpus[0].device_type)

GPUs disponibles: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU detectada: /physical_device:GPU:0
Tipo de device: GPU


In [3]:
if gpus:
    with tf.device('/GPU:0'):
        a = tf.random.normal([5000, 5000])
        b = tf.random.normal([5000, 5000])
        c = tf.matmul(a, b)
        print('Matmul (5000x5000) ejecutado en GPU')
        print('Resultado shape:', c.shape)
        print('Valor medio:', tf.reduce_mean(c).numpy())
else:
    print('No hay GPU, probando en CPU...')
    a = tf.random.normal([5000, 5000])
    b = tf.random.normal([5000, 5000])
    c = tf.matmul(a, b)
    print('Resultado shape:', c.shape)

I0000 00:00:1780355175.689818   29450 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3586 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Matmul (5000x5000) ejecutado en GPU
Resultado shape: (5000, 5000)
Valor medio: 0.006419576


In [4]:
if gpus:
    with tf.device('/GPU:0'):
        # Entrenar un modelo tiny para verificar que todo el pipeline funciona
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation='relu'),
            tf.keras.layers.Dense(10, activation='softmax')
        ])
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
        x = tf.random.normal([1024, 128])
        y = tf.random.uniform([1024], maxval=10, dtype=tf.int32)
        model.fit(x, y, epochs=3, verbose=2)
        print('--- Pipeline completo verificado en GPU ---')

Epoch 1/3


I0000 00:00:1780355176.876298   29607 service.cc:153] XLA service 0x7a5660044260 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780355176.876341   29607 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6 (Driver: 12.5.0; Runtime: 12.5.0; Toolkit: 12.5.0; DNN: 9.23.0)
I0000 00:00:1780355176.933629   29607 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780355177.127003   29607 cuda_dnn.cc:461] Loaded cuDNN version 92300
I0000 00:00:1780355177.139211   29607 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_928__.9
I0000 00:00:1780355177.177050   29607 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set ins

32/32 - 4s - 114ms/step - loss: 2.5051
Epoch 2/3


I0000 00:00:1780355179.735482   29607 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


32/32 - 0s - 4ms/step - loss: 2.0897
Epoch 3/3
32/32 - 0s - 4ms/step - loss: 1.8421
--- Pipeline completo verificado en GPU ---
